# Timeseries Module — Demo Notebook

Demonstrates the `timeseries_builder.py` and `timeseries_backfill.py` modules.

| Module | Classes |
|---|---|
| `timeseries_builder.py` | `BaseTimeseriesBuilder` → `BelchatowTimeseriesBuilder`, `RybnikChwalowiceTimeseriesBuilder` |
| `timeseries_backfill.py` | `BackfillDownloader`, `BackfillTimeseriesBuilder`, `BackfillCoverageRepairer` |

**Full pipeline run requires:** CDSE credentials + CH4Net v8 weights + `data/npy_cache/` populated.

**Parts 1–4 below** work without data files.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root.name not in ('methane-api', '') and repo_root != repo_root.parent:
    repo_root = repo_root.parent
scripts_dir = repo_root / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

print(f'Repo root: {repo_root}')

---
## Part 1 — Class Configuration

Show site-specific constants from each builder without running any pipeline.

In [ ]:
from timeseries.timeseries_builder import (
    BelchatowTimeseriesBuilder,
    RybnikChwalowiceTimeseriesBuilder,
)

belchatow = BelchatowTimeseriesBuilder(log_to_file=False)
rybnik    = RybnikChwalowiceTimeseriesBuilder(log_to_file=False)

for builder in [belchatow, rybnik]:
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  {type(builder).__name__}')
    print(sep)
    print(f'  SiteName          : {builder.SiteName}')
    print(f'  TileId            : {builder.TileId}')
    print(f'  SiteLat / SiteLon : {builder.SiteLat} / {builder.SiteLon}')
    print(f'  DetectionThreshold: {builder.DetectionThreshold}')
    print(f'  ConformalTau      : {builder.ConformalTau}')
    print(f'  ScOffsets (N/S/E/W): {builder.ScOffsets}')
    print(f'  MinePolygon       : {len(builder.MinePolygon)} vertices')
    for lat, lon in builder.MinePolygon:
        print(f'    ({lat:.4f}, {lon:.4f})')
    print(f'  OutJson           : {builder.OutJson.relative_to(repo_root)}')

---
## Part 2 — Inspect Existing Bełchatów Time Series

Load the JSON if it already exists and summarise detection statistics.

In [ ]:
import json
import numpy as np

belchatow_json = repo_root / 'results_analysis' / 'belchatow_annual_timeseries.json'

if belchatow_json.exists():
    with open(belchatow_json) as f:
        store = json.load(f)

    records = store.get('records', [])
    summary = store.get('summary', {})

    print(f"Belchatow time series : {len(records)} records")
    print(f"Years covered         : {store.get('years', [])}")
    print()

    by_status = {}
    for r in records:
        q  = r.get('quantification', {})
        st = q.get('status', 'no_detection') if q else 'no_quant'
        by_status[st] = by_status.get(st, 0) + 1
    print("Quantification status counts:")
    for k, v in sorted(by_status.items()):
        print(f"  {k:<30} {v:>4}")

    detected = [r for r in records
                if r.get('quantification', {}).get('status') == 'quantified']
    if detected:
        flows = [r['quantification']['flow_rate_kgh'] for r in detected
                 if r['quantification'].get('flow_rate_kgh') is not None]
        print(f"\nDetected acquisitions  : {len(detected)}")
        print(f"Flow range             : {min(flows):.0f}-{max(flows):.0f} kg/h")
        print(f"Mean flow              : {np.mean(flows):.0f} kg/h")
    if summary.get('annualised_t_per_yr_upper'):
        print(f"Annualised (upper)     : {summary['annualised_t_per_yr_upper']:.0f} t/yr")
    for yr, t in sorted(summary.get('ct_ch4_annual_t_by_year', {}).items()):
        print(f"CT CH4 {yr}             : {t:,.1f} t/yr")

    builder_b = BelchatowTimeseriesBuilder(log_to_file=False)
    builder_b.PrintSummary(store)
else:
    print(f"No existing file at {belchatow_json.name}")
    print("Run BelchatowTimeseriesBuilder().Run() with full data to generate it.")

---
## Part 3 — Backfill Time Series and Coverage Repair

In [ ]:
from timeseries.timeseries_backfill import (
    BackfillTimeseriesBuilder,
    BackfillCoverageRepairer,
    BACKFILL_SITES,
)

backfill_json = repo_root / 'results_analysis' / 'historical_backfill_timeseries.json'

if backfill_json.exists():
    with open(backfill_json) as f:
        timeseries = json.load(f)

    total_ok     = sum(1 for recs in timeseries.values()
                       for r in recs if r.get('status') == 'ok')
    total_no_cov = sum(1 for recs in timeseries.values()
                       for r in recs if r.get('status') == 'no_coverage')
    print(f"Backfill time series: {len(timeseries)} sites")
    print(f"  OK records          : {total_ok}")
    print(f"  No-coverage records : {total_no_cov}")

    builder_ts = BackfillTimeseriesBuilder(log_to_file=False)
    builder_ts.PrintSummary(timeseries)
else:
    print(f"No existing file at {backfill_json.name}")
    print("Run: BackfillDownloader().Run() -> BackfillTimeseriesBuilder().Run()")

In [ ]:
# Demonstrate coverage repair logic on synthetic records (no data files needed)
repairer = BackfillCoverageRepairer()

degenerate_rec = {
    'site': 'weisweiler', 'acquisition_date': '2021-07-15',
    'status': 'ok',
    'site_mean': 0.425702, 'ctrl_mean': 0.425702,
    'sc_ratio': 1.0,       'cv_ctrl': 0.0,
    'cfar_detect': None,   'sc_cfar': None,
    'npy': None, 'tif': None,
}
healthy_rec = {
    'site': 'weisweiler', 'acquisition_date': '2022-07-01',
    'status': 'ok',
    'site_mean': 0.48,    'ctrl_mean': 0.35,
    'sc_ratio': 1.37,     'cv_ctrl': 0.12,
    'cfar_detect': True,  'sc_cfar': 1.21,
    'npy': None, 'tif': None,
}

print(f"degenerate_rec IsDegenerate: {repairer._IsDegenerate(degenerate_rec)}")
print(f"healthy_rec    IsDegenerate: {repairer._IsDegenerate(healthy_rec)}")

synthetic_ts = {'weisweiler': [degenerate_rec.copy(), healthy_rec.copy()]}
_, n_checked, n_fixed = repairer.Repair(synthetic_ts, dry_run=True)
print(f"\nRepair (dry-run): checked={n_checked}, would-fix={n_fixed}")
print(f"degenerate status unchanged (dry run): {synthetic_ts['weisweiler'][0]['status']}")

---
## Part 4 — Annualisation Arithmetic

The shared `_AnnualisationFields` helper computes three emission bounds from
detected flow rates and month-level observation coverage.

In [ ]:
from timeseries.timeseries_builder import BaseTimeseriesBuilder

rng = np.random.default_rng(42)

# Simulate 12 detections spread across 7 months (2 acquisitions in some months)
synthetic_flows   = rng.uniform(800, 1600, 12).tolist()
synthetic_detected = [
    {'quantification': {'status': 'quantified', 'flow_rate_kgh': f}}
    for f in synthetic_flows
]
det_months = [f'2024-{m:02d}' for m in [2, 4, 5, 6, 7, 8, 10]]
obs_months = [f'2024-{m:02d}' for m in range(1, 13)]

fields = BaseTimeseriesBuilder._AnnualisationFields(
    detected=synthetic_detected,
    det_months=det_months,
    obs_months=obs_months,
)

print('Synthetic annualisation (12 detections, 7/12 months observed positive):')
print(f"  Mean flow rate               : {fields['mean_flow_kg_h']:.0f} kg/h")
print(f"  Detection months / observed  : {fields['n_detection_months']} / {fields['n_observed_months']}")
print(f"  Upper bound                  : {fields['annualised_t_per_yr_upper']:.0f} t/yr")
print(f"  Floor-imputed (Q_floor={fields['detection_floor_kgh_assumed']:.0f} kg/h): "
      f"{fields['annualised_t_per_yr_floor_imputed']:.0f} t/yr")
print(f"  Lower bound                  : {fields['annualised_t_per_yr_lower_bound']:.0f} t/yr")

# Verify arithmetic
mean_q = fields['mean_flow_kg_h']
expected_upper = mean_q * 8760 / 1000
ok = abs(fields['annualised_t_per_yr_upper'] - expected_upper) < 1
print(f"\nVerification: upper = {mean_q:.1f} * 8760/1000 = {expected_upper:.0f} t/yr  [{'OK' if ok else 'FAIL'}]")

---
## Part 5 — Run Checklist

| Task | Requires | Code |
|---|---|---|
| Download 2020–2023 tiles | CDSE credentials | `BackfillDownloader().Run()` |
| Compile backfill S/C | CH4Net weights | `BackfillTimeseriesBuilder().Run()` |
| Repair partial-swath records | backfill JSON | `BackfillCoverageRepairer().Run()` |
| Bełchatów annual series | CDSE + CH4Net | `BelchatowTimeseriesBuilder().Run(years=[2024])` |
| Rybnik annual series | CDSE + CH4Net | `RybnikChwalowiceTimeseriesBuilder().Run(years=[2024])` |

All classes support `dry_run=True` (catalogue search only) and incremental resume.